# SOM (Self-Organizing Map) sur MNIST

Même structure que les notebooks précédents, adaptée à la SOM de Kohonen. Comme K-Means, l'espace latent est discret (indice du BMU), mais la SOM ajoute une structure topologique : les neurones sont disposés sur une grille 2D, ce qui permet des visualisations spécifiques (U-Matrix, Hit Map, projection colorée).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("__file__"), "..")))

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.dataset import load_mnist_dataset
from src.helper import extract_full_dataset
from src.som import SOM
from src.metrics import compression_report, Latent

## 1. Chargement des données

Même sous-ensemble que les autres notebooks, pour que les comparatifs restent valides.

In [2]:
dataloader = load_mnist_dataset(train=True, batch_size=4096, download=True, shuffle=False)
images, digit_labels = extract_full_dataset(dataloader)

N = 10000
X = images[:N].flatten(start_dim=1).numpy()
digit_labels = digit_labels[:N].numpy()

print("X shape:", X.shape)

X shape: (10000, 784)


## 2. Choix de la taille de grille

On entraîne la SOM avec plusieurs tailles de grille pour évaluer l'impact sur la qualité de quantification et la préservation topologique. La quantization error mesure la fidélité de la reconstruction, la topographic error mesure la continuité de la carte.

In [ ]:
GRID_SHAPES = [(4, 4), (8, 8), (10, 10), (16, 16), (18, 18)]

sweep_results = []
for gs in GRID_SHAPES:
    print(f"Training SOM {gs[0]}x{gs[1]} ({gs[0]*gs[1]} neurons)...")
    som = SOM(grid_shape=gs, n_iterations=5000, random_state=42)
    som.fit(X)
    qe = som.quantization_error(X)
    te = som.topographic_error(X)
    sweep_results.append((gs, qe, te))
    print(f"  -> QE = {qe:.4f}, TE = {te:.4f}")

Training SOM 4x4 (16 neurons)...
  -> QE = 5.9842, TE = 0.3905
Training SOM 8x8 (64 neurons)...
  -> QE = 5.4569, TE = 0.3931
Training SOM 10x10 (100 neurons)...
  -> QE = 5.3012, TE = 0.4085
Training SOM 16x16 (256 neurons)...
  -> QE = 5.0351, TE = 0.3924


In [4]:
labels = [f"{gs[0]}x{gs[1]}\n({gs[0]*gs[1]})" for gs, _, _ in sweep_results]
qes = [r[1] for r in sweep_results]
tes = [r[2] for r in sweep_results]

fig = make_subplots(rows=1, cols=2, subplot_titles=["Quantization Error", "Topographic Error"])

fig.add_trace(
    go.Bar(x=labels, y=qes, marker_color="#636efa", name="QE"),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=labels, y=tes, marker_color="#EF553B", name="TE"),
    row=1, col=2
)

fig.update_layout(
    title_text="Impact de la taille de grille sur les métriques SOM",
    template="plotly_dark",
    showlegend=False,
    height=400, width=900
)
fig.show()

## 3. Fit avec la grille choisie

On retient une grille 10×10 (100 neurones) comme bon compromis entre résolution et temps de calcul.

In [5]:
model = SOM(grid_shape=(10, 10), n_iterations=5000, random_state=42)
model.fit(X)

print(f"Grid: {model.grid_shape}")
print(f"Neurons: {model.n_neurons}")
print(f"Quantization Error: {model.quantization_error(X):.4f}")
print(f"Topographic Error: {model.topographic_error(X):.4f}")

Grid: (10, 10)
Neurons: 100
Quantization Error: 5.3012
Topographic Error: 0.4085


## 4. Courbe de convergence

La quantization error est échantillonnée périodiquement pendant l'entraînement. La décroissance monotone confirme la convergence de l'algorithme.

In [6]:
fig = px.line(
    x=list(range(len(model.history_))),
    y=model.history_,
    labels=dict(x="Échantillon (itération / log_interval)", y="Quantization Error"),
    title="Convergence de la SOM (Quantization Error pendant l'entraînement)"
)
fig.update_layout(template="plotly_dark", width=800, height=400)
fig.show()

## 5. Encode / decode / rapport de compression

Même pipeline que K-Means : chaque image est encodée par l'indice de son BMU, décodée par le vecteur de poids du BMU correspondant.

In [7]:
latent = model.encode(X)
X_reconstructed = model.decode(latent)
codebook = model.get_codebook()

report = compression_report(codebook, latent, X, X_reconstructed)
for k, v in report.items():
    print(f"{k}: {v}")

latent_nature: discrete
codebook_bytes: 313600
latent_bytes: 10000
total_compressed_bytes: 323600
original_bytes: 31360000
compression_ratio: 96.9097651421508
reconstruction_mse: 0.037563495337963104


## 6. Visualisation des neurones (codebook)

Chaque neurone possède un vecteur de poids W de dimension 784. On le reshape en (28, 28) pour l'afficher comme une image, disposée selon la position du neurone sur la grille.

In [8]:
rows, cols = model.grid_shape

fig = make_subplots(
    rows=rows, cols=cols,
    horizontal_spacing=0.01,
    vertical_spacing=0.01
)

for idx in range(model.n_neurons):
    r = idx // cols + 1
    c = idx % cols + 1
    neuron_img = model.W[idx].reshape(28, 28)
    fig.add_trace(
        go.Heatmap(z=neuron_img, colorscale="gray", showscale=False),
        row=r, col=c
    )

for i in range(1, rows * cols + 1):
    fig.update_yaxes(autorange="reversed", showticklabels=False, row=(i-1)//cols+1, col=(i-1)%cols+1)
    fig.update_xaxes(showticklabels=False, row=(i-1)//cols+1, col=(i-1)%cols+1)

fig.update_layout(
    height=80 * rows, width=80 * cols,
    title_text="Neurones de la SOM (prototypes appris)",
    template="plotly_dark"
)
fig.show()

## 7. Visualisation : original vs reconstruit

In [9]:
np.random.seed(42)
indices = np.random.choice(X.shape[0], size=5, replace=False)

fig = make_subplots(
    rows=2, cols=5,
    subplot_titles=[f"Original {idx}" for idx in indices] + [f"Reconstructed {idx}" for idx in indices]
)

for i, idx in enumerate(indices):
    orig_img = X[idx].reshape(28, 28)
    recon_img = X_reconstructed[idx].reshape(28, 28)

    fig.add_trace(go.Heatmap(z=orig_img, colorscale="gray", showscale=False), row=1, col=i+1)
    fig.add_trace(go.Heatmap(z=recon_img, colorscale="gray", showscale=False), row=2, col=i+1)

fig.update_layout(
    height=500, width=1000,
    title_text="SOM : Original vs Reconstructed",
    template="plotly_dark"
)

for row in [1, 2]:
    for col in range(1, 6):
        fig.update_yaxes(autorange="reversed", showticklabels=False, row=row, col=col)
        fig.update_xaxes(showticklabels=False, row=row, col=col)

fig.show()

## 8. Hit Map

La Hit Map comptabilise le nombre d'échantillons assignés à chaque neurone. Elle permet de détecter les neurones « morts » (jamais sollicités) et les zones de la carte qui concentrent les données.

In [11]:
bmu_indices = latent.array
hit_counts = np.zeros(model.n_neurons, dtype=int)
for idx in bmu_indices:
    hit_counts[idx] += 1

hit_map = hit_counts.reshape(model.grid_shape)

fig = go.Figure(data=go.Heatmap(
    z=hit_map,
    colorscale="Blues",
    colorbar=dict(title="Nombre de hits"),
    text=hit_map,
    texttemplate="%{text}",
    textfont=dict(size=8)
))
fig.update_layout(
    title="Hit Map (nombre d'échantillons par neurone)",
    template="plotly_dark",
    width=600, height=550,
    yaxis=dict(autorange="reversed")
)
fig.show()

n_dead = int(np.sum(hit_counts == 0))
print(f"Neurones morts (0 hits) : {n_dead} / {model.n_neurons}")

Neurones morts (0 hits) : 0 / 100


## 9. Projection 2D colorée par digit

On place chaque échantillon à la position (row, col) de son BMU sur la grille, avec un léger jitter pour éviter la superposition. La couleur indique le chiffre réel : si la carte préserve bien la topologie, les chiffres similaires devraient se regrouper.

In [12]:
# Position grille de chaque BMU + jitter pour la lisibilité
rng = np.random.default_rng(42)
bmu_coords = model.C[latent.array]
jitter = rng.uniform(-0.3, 0.3, size=bmu_coords.shape)
coords_jittered = bmu_coords + jitter

fig = px.scatter(
    x=coords_jittered[:, 1],
    y=coords_jittered[:, 0],
    color=digit_labels.astype(str),
    color_discrete_sequence=px.colors.qualitative.Plotly,
    labels=dict(x="Colonne de la grille", y="Ligne de la grille", color="Chiffre Réel"),
    title="SOM : Projection des échantillons sur la grille (colorée par digit)"
)
fig.update_layout(
    width=800, height=700,
    template="plotly_dark",
    yaxis=dict(autorange="reversed")
)
fig.update_traces(marker=dict(size=3, opacity=0.6))
fig.show()

## 10. Génération de nouvelles images par sampling des neurones

Comme pour PCA, on génère de nouvelles images en sampling l'espace latent des neurones.
Ici, le latent est discret (indice du neurone), donc on sample des neurones existants ou proches.

Deux approches :
1. Afficher les neurones avec le plus d'activité (hits élevés) - ce sont les prototypes appris.
2. Interpoler entre deux neurones voisins pour obtenir une image "entre" deux prototypes.

In [ ]:
bmu_indices = latent.array
hit_counts = np.zeros(model.n_neurons, dtype=int)
for idx in bmu_indices:
    hit_counts[idx] += 1

# Trouver les neurones avec le plus de hits
top_neuron_indices = np.argsort(hit_counts)[-8:][::-1]  # Top 8

fig = make_subplots(rows=2, cols=4, subplot_titles=[
    f"Neurone {idx}\n(hits: {hit_counts[idx]})" for idx in top_neuron_indices
])

for i, neuron_idx in enumerate(top_neuron_indices):
    row = i // 4 + 1
    col = i % 4 + 1
    
    neuron_img = model.W[neuron_idx].reshape(28, 28)
    
    fig.add_trace(
        go.Heatmap(z=neuron_img, colorscale="gray", showscale=False),
        row=row, col=col
    )

fig.update_layout(
    height=500, width=1200,
    title_text="Les 8 neurones les plus actifs (prototypes importants)",
    template="plotly_dark"
)
for i in range(1, 9):
    fig.update_yaxes(autorange="reversed", showticklabels=False, row=(i-1)//4+1, col=(i-1)%4+1)
    fig.update_xaxes(showticklabels=False, row=(i-1)//4+1, col=(i-1)%4+1)
fig.show()

print(f"Hits des neurones sélectionnés : {[hit_counts[idx] for idx in top_neuron_indices]}")